# QC generated summaries

Created by: Sahana Kowshik

In [2]:
import pandas as pd
import re
import json

In [3]:
def print_visit_summary(df, index):
    print(df.iloc[index]['visit_summary'])
    
def print_patient_summary(df, index):
    print(json.dumps(json.loads(df.iloc[index]['patient_summary']), indent=4))

In [130]:
# Define the path to the CSV you want to load:
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv"
path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv"

# Read the data
summaries = pd.read_csv(path)
len(summaries)

5812

In [131]:
len(summaries)

5812

In [132]:
summaries.head()

,NACCID,NACCVNUM,NACCADC,PACKET,FORMVER,VISITMO,VISITDAY,VISITYR,NACCAVST,NACCNVST,...,RTRTEM,RTRTEMM,NACCDICO,sort_key,sort_year,patient_summary,diag_summary,VISITGAP,visit_summary_prompt,visit_summary
0,NACC772223,1,5310,I,2.0,1,29,2009,1,1,...,NaN,NaN,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",1,<|im_start|>user\nYou will receive patient dat...,"The subject is a 78-year-old White, right-hand..."
1,NACC900564,5,8974,F,2.0,12,2,2009,5,5,...,NaN,NaN,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",2,<|im_start|>user\nYou will receive patient dat...,"The subject is an 89-year-old right-handed, En..."
2,NACC586854,4,8658,T,3.0,8,23,2016,4,2,...,NaN,NaN,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",0,<|im_start|>user\nYou will receive patient dat...,The subject is a 63-year-old female who visite...
3,NACC474857,3,2096,F,2.0,3,23,2010,3,3,...,NaN,NaN,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",2,<|im_start|>user\nYou will receive patient dat...,The subject is a 67-year-old male of White rac...
4,NACC316274,9,5783,F,2.0,10,1,2014,9,9,...,NaN,NaN,NaN,0,NaN,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",1,<|im_start|>user\nYou will receive patient dat...,"The subject is a 103-year-old right-handed, En..."


In [133]:
print_visit_summary(summaries, -10)

The patient is a 79-year-old White, right-handed female with 17 years of education, who is married and lives with her spouse in a private residence. She is completely dependent in terms of her level of independence. The primary reason for her visit to the Alzheimer’s Disease Center (ADC) is to participate in a research study, and she was referred by a professional contact, such as a clinician, nurse, or other healthcare provider. She does not identify as Hispanic/Latino and speaks English as her primary language.

In terms of family history, the patient reports having at least one first-degree family member with cognitive impairment, although no specific mention of her mother or father having cognitive impairment is made. The patient is currently taking four medications: Vitamin E, Donepezil, Risedronate, and Memantine. These include FDA-approved medications for Alzheimer’s disease symptoms, indicating some concern for cognitive decline despite the absence of reported active depression

In [134]:
# summaries['Q_TYPE'].value_counts()

In [ ]:
print("Checking if amyloid is present in amyloid cases")
x = summaries[(summaries['Q_TYPE'] == 'Amyloid PET')]
print(len(x))
cnt = 0
for i, row in x.iterrows():
    if ('Abnormally elevated amyloid on PET' in row['patient_summary']):
        print(i, "ERROR available in patient_summary")
    if 'amyloid ' in row['visit_summary'].lower():
        print(i, "ERROR available in visit_summary")
        cnt += 1
    
    # if cnt > 5:
    #     break
    
x = summaries[(summaries['Q_TYPE'] == 'Amyloid CSF')]
print(len(x))
cnt = 0
for i, row in x.iterrows():
    if ('Abnormally low amyloid in CSF' in row['patient_summary']):
        print(i, "ERROR available in patient_summary")
    if 'amyloid ' in row['visit_summary'].lower():
        print(i, "ERROR available in visit_summary")
        cnt += 1
    
    # if cnt > 5:
    #     break
    

In [9]:
print("Checking if amyloid is not present in other cases")
x = summaries[~(summaries['Q_TYPE'].isin(['Amyloid PET', 'Amyloid CSF']))]
print(len(x))
cnt = 0
for i, row in x.iterrows():
    if ('Abnormally elevated amyloid on PET' in row['patient_summary']) | ('Abnormally low amyloid in CSF' in row['patient_summary']):
        # print(i, "ERROR NOT available in patient_summary")
        if 'amyloid' not in row['visit_summary'].lower():
            print(i, "ERROR NOT available in visit_summary")
            cnt += 1
            # break
    
    # if cnt > 5:
    #     break
    
# x = summaries[(summaries['Q_TYPE'] != 'Amyloid CSF')]
# print(len(x))
# cnt = 0
# for i, row in x.iterrows():
#     if ('Abnormally low amyloid in CSF' in row['patient_summary']):
#         print(i, "ERROR available in patient_summary")
#     if 'amyloid ' in row['visit_summary'].lower():
#         print(i, "ERROR available in visit_summary")
#         cnt += 1
    
#     # if cnt > 5:
#     #     break
            

Checking if amyloid is not present in other cases
40264
18185 ERROR NOT available in visit_summary
37663 ERROR NOT available in visit_summary


In [135]:
import pandas as pd
import re
from collections import Counter

def find_repeated_sentences_in_text(text):
    # Split into sentences using punctuation (adjust if needed)
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    normalized = [s.strip().lower() for s in sentences if s.strip()]
    counter = Counter(normalized)
    repeated = {sent: count for sent, count in counter.items() if count > 1}
    return repeated

repeats_info = []

for idx, row in summaries.iterrows():
    text = row['visit_summary']
    repeats = find_repeated_sentences_in_text(text)
    if repeats:
        for sentence, count in repeats.items():
            if count > 2:
                repeats_info.append({
                    "row_index": idx,
                    "sentence": sentence,
                    "count": count
                })

# Convert results to a DataFrame for easy analysis
repeats_df = pd.DataFrame(repeats_info)

In [136]:
repeats_df

""


In [137]:
if 'row_index' in repeats_df:
    repeat_list = set(repeats_df['row_index'])
else:
    repeat_list = set()

In [138]:
repeat_list

set()

In [139]:
summaries_dropped_index = summaries.drop(repeat_list).reset_index(drop=True)

In [140]:
# dropped = summaries.loc[list(repeat_list)]

In [141]:
# summaries.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv", index=False)

In [142]:
# summaries_dropped_index = summaries

In [143]:
ages = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    ages.append(pat['Subject Demographics']["Subject's age at visit"])

In [144]:
index = -1
incorrect_age_count = []
incorrect_cop_age_count = []

# Define regex pattern to extract and replace "a/an n-year-old"
age_pattern = r"\b(\d{1,3})-year-old\s\b"

def remove_incorrect_age(text):
    global count
    global index
    global incorrect_cop_age
    index += 1
    # Process text line by line
    modified_text = []
    for sentence in text.split("."):
        match = re.search(age_pattern, sentence)
        if match:  # Check if the sentence matches the pattern
            # print(sentence)
            age = match.group(1)  # Extract age
            # if "co-participant" in sentence.lower() or "spouse" in sentence.lower() or "child" in sentence.lower() or "daughter" in sentence.lower() or "friend" in sentence.lower() or "son" in sentence.lower() or "sibling" in sentence.lower() or "paid caregiver" in sentence.lower() or "relative" in sentence.lower():
            #     incorrect_cop_age_count.append((index, age))
                
            # else:   
            #     if int(age) != int(ages[index]):
            #         incorrect_age_count.append((index, age))
                    
            if int(age) != int(ages[index]) and 'co-participant' not in sentence:
                # print(text)
                incorrect_age_count.append((index, age, match, ages[index]))
            
    
    # return final_text

summaries_dropped_index["visit_summary"].apply(remove_incorrect_age)
print("Total sentences with incorrect co participant age:", len(incorrect_cop_age_count))
print("Total sentences with incorrect age:", len(incorrect_age_count))

Total sentences with incorrect co participant age: 0
Total sentences with incorrect age: 0


In [145]:
incorrect_age_count

[]

In [146]:
smoke_quit_age = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    try:
        smoke_quit_age.append(pat['Subject Health History']["If the subject quit smoking, age at which he/she last smoked (i.e., quit) (range: (7.0 - 110.0))"])
    except:
        smoke_quit_age.append(-1)
        # pass

In [147]:
smoke_quit_age[200]

30.0

In [148]:
import re

# Define regex pattern to identify relevant sentences
pattern = r"\b\d{1,3}-year-old individual\b.*?\bsmoking\b|\bsmoking\b.*?\b\d{1,3}-year-old\b"

# Define regex pattern to extract and replace "a/an n-year-old"
age_pattern_1 = r"\b(\d{1,3})-year-old\s\b"
age_pattern_2 = r"\bat age (\d{1,3})\s\b"

# Regex pattern to match "The patient quit smoking at the age of X"
quit_smoking_pattern = r"The patient quit smoking at the age of (\d+)"

In [149]:
index = 0
incorrect_smoking_age_count = []
invalid_smoking_age_count = []
quit_30_days_count = []

def clean_smoking_part(text):
    global count
    global index
    global invalid_smoking_age
    global quit_30_days_count
    
    modified_text = []
    
    for sentence in text.split("."):
        # sentence = sentence.strip()  # Remove leading/trailing spaces
        
        if "They quit smoking 30 days prior to the initial visit" in sentence:
            # sentence = sentence.replace("They quit smoking 30 days prior to the initial visit", "They did not smoke cigarettes for 30 days prior to the visit")
            quit_30_days_count.append(index)
        
        # Check for "The patient quit smoking at the age of X"
        quit_match = re.search(quit_smoking_pattern, sentence)
        if quit_match:
            age = int(quit_match.group(1))
            if int(age) > int(ages[index]):  
                invalid_smoking_age_count.append(index)
                continue

        # Check for "n-year-old" + "smoking"
        match = re.search(age_pattern_1, sentence)
        # if match and re.search(pattern, sentence):  
        if match and "smok" in sentence.lower(): 
            age = match.group(1)  # Extract age
            print(age)
            raise
            if int(age) != int(smoke_quit_age[index]):
                # sentence = re.sub(remove_pattern, "", sentence).strip()   # Replace "n-year-old" with ""
                incorrect_smoking_age_count.append((index, match))
        
        match = re.search(age_pattern_2, sentence)
        # if match and re.search(pattern, sentence):  
        if match and "smok" in sentence.lower(): 
            age = match.group(1)  # Extract age
            # print(age)
            # raise
            if int(age) != int(smoke_quit_age[index]):
                # sentence = re.sub(remove_pattern, "", sentence).strip()   # Replace "n-year-old" with ""
                incorrect_smoking_age_count.append((index, match))
            
            # if int(age) <= int(ages[index]) and (int(age) == int(smoke_quit_age[index]) and smoke_quit_age[index] != -1):
            #     # new_sentence = f"\n\nThe patient quit smoking at the age of {age}. "  # Create new sentence
            #     # modified_text.append(new_sentence + sentence)  # Add new sentence before original
            #     incorrect_smoking_age_count.append((index, match))
            # else:
            #     modified_text.append("\n\n" + sentence)
                # print(index)
                # raise ValueError
        # else:
        #     modified_text.append(sentence)  # Keep other sentences unchanged

    # Join sentences back into a full text
    # final_text = ".".join(modified_text)  # Removes any accidental empty sentences
    index += 1
    # return final_text

# Apply function to DataFrame column
summaries_dropped_index["visit_summary"].apply(clean_smoking_part)

print("Incorrect smoking age:", len(incorrect_smoking_age_count))
print("Invalid smoking age:", len(invalid_smoking_age_count))
print("Invalid quit_smoking_days:", len(quit_30_days_count))


Incorrect smoking age: 0
Invalid smoking age: 0
Invalid quit_smoking_days: 0


In [150]:
incorrect_smoking_age_count

[]

In [25]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [26]:
len(tokenizer.encode(summaries.iloc[1883]['visit_summary']))

1024

In [27]:
max_len = 0
for i, row in summaries.iterrows():
    max_len = max(max_len, len(tokenizer.encode(row['visit_summary'])))
    
max_len

1760

In [157]:
x = "You are a helpful AI assistant that provides well-reasoned and detailed responses. First, think step by step through the reasoning process as an internal monologue, and then provide the user with the final answer within \\boxed{}. Respond using the following format: <think>\n...\n</think>\n<answer>\n...\n</answer>, i.e., the reasoning process should be enclosed within <think>...</think> tags, and the final answer within <answer>...</answer> tags. Reply in English only, do not use other languages."

In [158]:
print(x)

You are a helpful AI assistant that provides well-reasoned and detailed responses. First, think step by step through the reasoning process as an internal monologue, and then provide the user with the final answer within \boxed{}. Respond using the following format: <think>
...
</think>
<answer>
...
</answer>, i.e., the reasoning process should be enclosed within <think>...</think> tags, and the final answer within <answer>...</answer> tags. Reply in English only, do not use other languages.
